In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split

I0000 00:00:1779546563.425485    5467 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
IMG_SIZE = 256  # Standardized input resolution for both phases
BATCH_SIZE = 4 # Batch size
EPOCHS_PHASE1 = 100  # Semantic Segmentation Focus, for Segmentation 
EPOCHS_PHASE2 = 30   # Segmentation-Based Classification Focus for classification Polyp and Non-Polyp

#### Declaring dataset path for segmentation and classification

In [3]:
# Directory and Artifact Paths
DATA_PATH = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2'
POSITIVE_DIR = os.path.join(DATA_PATH, 'imagesAll_positive') # For Polyp classification
NEGATIVE_DIR = os.path.join(DATA_PATH, 'sequenceData/negativeOnly')  # For non polyp classification
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/unified_net/thesis_v3_sequential.keras"

#### mixed_precision
Activate mixed precision training for efficiency and memory optimization
It is a performance optimization for GPU. It utilize half of the GPU Memory, 
increases the the model training speed 3 times without effecting on accuracy. 
mixed_float16, 16 bits helps to get speedy calculation when need, Flawless accuracy performed by 32-bits

In [4]:

mixed_precision.set_global_policy('mixed_float16')

#### Image Augmentation
As we have limited datasources we have to augment images. If we train with limited datasets Model can memorize 
instead of learning which causes overfitting. An overfitted model cannot predict other image except training dataset. Even it cannot predict on validation dataset as well.The **overfitting occurs** when The Training Loss going down but the validation loss increases at a time

In [5]:

def augment(img, mask):
    """Applies shape-preserving geometric and radiometric transformations."""
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask

#### Data / Image pre-processing pipeline for segmentation
This functions primary task is to load raw images corresponding its mask image from the file storage, clean them up, standardizing the image size finally convert them into clean tensor ( or a 2D matrix) suitable for the model to be trained.


In [6]:
def process_path_segmentation(img_path, mask_path):
    """Preprocesses input frames and localized binary masks for Dataset B."""
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.cast(mask, tf.float32) / 255.0
    mask = tf.where(mask > 0.5, 1.0, 0.0)
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask

#### Data / Image Pre-processing for classification
getting the image and label wheather it is polyp or not, also converting the image for standardization

In [8]:
def process_path_classification(img_path, label):
    """Preprocesses input tensors and categorical targets for Dataset A."""
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

#### Load Segmentation data path
Instead of loading the actual images into memory, it loads the images path dynamically, 
The image is only collected for segmentation training, it it's exact mask match in the dataset directory.

In [10]:
def load_segmentation_paths(base_path):
    """Extracts paired image and mask absolute paths from multi-center repositories."""
    all_images, all_masks = [], []
    for i in range(1, 7):
        center_id = f"C{i}"
        img_dir = os.path.join(base_path, f"data_{center_id}", f"images_{center_id}")
        mask_dir = os.path.join(base_path, f"data_{center_id}", f"masks_{center_id}")
        if os.path.exists(img_dir) and os.path.exists(mask_dir):
            img_paths = (glob.glob(os.path.join(img_dir, "*.jpg")) + glob.glob(os.path.join(img_dir, "*.JPG")) +
                         glob.glob(os.path.join(img_dir, "*.png")) + glob.glob(os.path.join(img_dir, "*.jpeg")))
            for img_p in img_paths:
                filename = os.path.basename(img_p)
                name_without_ext, ext = os.path.splitext(filename)
                expected_mask_name = f"{name_without_ext}_mask{ext}"
                mask_p = os.path.join(mask_dir, expected_mask_name)
                if os.path.exists(mask_p):
                    all_images.append(img_p)
                    all_masks.append(mask_p)
    return all_images, all_masks

#### Load Classification data path
this function labels the images into two classes -0 for Negative image or Non-Polyp and 1 for Positive image or Polyp
**Note:** When your dataset has 90% negative and 10% positive, highly imbalance dataset it creates a cheat code to get high accuracy. thus it causes a Accuracy Paradox, and for this reason it misclassify the image. 

In [12]:
def load_classification_paths():
    """Compiles absolute paths for global positive and artifact-heavy negative sequences."""
    valid_exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.PNG', '*.JPG', '*.JPEG')
    pos_files, neg_files = [], []
    for ext in valid_exts:
        pos_files.extend(glob.glob(os.path.join(POSITIVE_DIR, "**", ext), recursive=True))
    for i in range(1, 14):
        target_folder = os.path.join(NEGATIVE_DIR, f"seq{i}_neg")
        if os.path.exists(target_folder):
            for ext in valid_exts:
                neg_files.extend(glob.glob(os.path.join(target_folder, "**", ext), recursive=True))
    pos_files = [f for f in pos_files if os.path.isfile(f)]
    neg_files = [f for f in neg_files if os.path.isfile(f)]
    
    file_paths = pos_files + neg_files
    labels = [1.0] * len(pos_files) + [0.0] * len(neg_files)
    return np.array(file_paths), np.array(labels, dtype=np.float32)

#### Squeeze-and-Excitation Block
To get channel-wise attention it is very much powerful for CNN. An SE_Block dynamically learns which channel/features are the most important for a specific image and scale up the specific features with dimming down the noise with irrelevent ones.
Standard CNN only looks at the small neighbour of the pixel at a time, But GlobalAveragePooling2D gets the average value of pixels.  from the total height and weight of a channel. 
Ho
We can consider an image with a 3D feature map, as it has three distinct geometric dimentions: Height, Width and Depth (Channels). 
Image Height (H) : Vertical Pixels (256 pixels height)
Image Width (W) : Horizontal Pixels (256 pixels width)
Channels (C): The depth layer, this is 3 (RGB)
So an image has a 3D block dimentions = 256x256x3. 
When an image passes through convolutional layers, the spatial dimentions (H and W) usually shrink, but the number of channels (C) expands dramatically, Deep in your network you might have a feature map of size 16x16x512 that is a literal 3D cube made of 131,072 numarial values.
** The Sqeeze Turns 3D into 1D** ```layers.GlobalAveragePooling2D()(input_tensor)``` this sentence takes the 3D Cube and flattens its spatial dimentions. 

```sh
3D CUBE[16x16x512] -----Global Average Pooling------> 1D vector [512]

Before Squeeze (3D Cube):
       _______
      /      /|
     /______/ |  <- 512 separate 16x16 slices stacked deep
    |       | |
    |_______|/
  16x16 spatial grid

After Squeeze (1D Array/Vector):
    [ 0.78,  0.12,  0.91,  0.03, ... , 0.55 ]  <- Exactly 512 numbers long
```

In [13]:
def se_block(input_tensor, ratio=16):
    """Channel-wise Squeeze-and-Excitation Attention Layer."""
    channels = input_tensor.shape[-1]
    squeeze = layers.GlobalAveragePooling2D()(input_tensor)
    excitation = layers.Dense(channels // ratio, activation='relu', use_bias=False)(squeeze)
    excitation = layers.Dense(channels, activation='sigmoid', use_bias=False)(excitation)
    excitation = layers.Reshape((1, 1, channels))(excitation)
    return layers.multiply([input_tensor, excitation])